# FLUX.2 Klein Remote GPU Inpainting Server

**Dedicated High-Performance Inpainting Backend for MangaTranslator**

Offload heavy manga speech bubble and outside-text inpainting to a free Google Colab Tesla T4 GPU or Kaggle environment.

### ✨ Key Capabilities
- **High-Performance Single Worker**: Dedicated pipeline leveraging full GPU memory bus bandwidth (~18–20s per inpaint pass on Tesla T4) with direct VAE decoding and lazy loading.
- **Zero Text-Encoder Architecture**: Bypasses the ~2.8 GB text encoder entirely by loading precomputed prompt embeddings (`flux4b_manga_embeds.pt`), slashing load time to ~2 seconds.
- **Zero-Config Cloudflare Tunnel**: Automatically provisions a secure public tunnel via Cloudflare (`trycloudflare.com`) with no port forwarding required.
- **Lossless 1:1 Inpainting**: Inpaints cropped clusters up to 1024x1024 natively, preserving fine screentones and line art.

---
### Step-by-Step Instructions
1. Switch to GPU runtime: **Runtime -> Change runtime type -> T4 GPU**.
2. Upload `flux4b_manga_embeds.pt` to your Google Drive root (`MyDrive/flux4b_manga_embeds.pt`) or let the script prompt you to upload it.
3. Run the code cell below (`Shift + Enter`).
4. Copy the generated Cloudflare URL (e.g., `https://xxxx.trycloudflare.com`).
5. In your local MangaTranslator Web UI (*Settings -> Outside Text Inpainting*):
   - Inpainting Method: `Flux.2 Klein 4B`
   - Backend: `Remote Colab`
   - Colab Server URL: Paste your Cloudflare URL
   - Click **Test Connection** to verify.

In [ ]:
MODEL_CHOICE = "flux_klein_4b"  # @param ["flux_klein_4b", "flux_klein_9b"]
HF_TOKEN = ""  # @param {type:"string"}

import os
import sys
import time
import gc
import re
import io
import asyncio
import subprocess
import urllib.request
from PIL import Image

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["DIFFUSERS_SDNQ_TRANSFORMERS"] = "1"
os.environ["DIFFUSERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)

if HF_TOKEN and HF_TOKEN.strip():
    token_val = HF_TOKEN.strip()
    if not token_val.startswith("hf_"):
        token_val = "hf_" + token_val
    os.environ["HF_TOKEN"] = token_val

os.system("pkill -9 cloudflared 2>/dev/null || true")

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception:
    pass

DRIVE_DIR = "/content/drive/MyDrive"

try:
    import sdnq
    import fastapi
    import uvicorn
    import diffusers
except ImportError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "fastapi", "uvicorn", "python-multipart",
        "diffusers>=0.33.0", "transformers", "accelerate", "sdnq>=0.2.2", "sentencepiece", "protobuf"
    ], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.")

gc.collect()
torch.cuda.empty_cache()
cur_vram = round(torch.cuda.memory_allocated(0) / (1024**3), 2)
if cur_vram > 2.0:
    print(f"⚠️ Warning: High initial VRAM ({cur_vram} GB). Restart session if OOM occurs.")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import diffusers.utils.import_utils as diff_import_utils
# Disable torchao & xformers checks to bypass Colab torchao FqnToConfig ImportError
diff_import_utils._xformers_available = False
diff_import_utils.is_xformers_available = lambda: False
diff_import_utils._torchao_available = False
diff_import_utils.is_torchao_available = lambda: False

import diffusers.models.attention_dispatch as attn_dispatch
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import Response, JSONResponse
from diffusers import Flux2KleinPipeline

def fast_full_attention(query, key, value, *args, **kwargs):
    orig_dtype = query.dtype
    needs_cast = (query.dtype == torch.bfloat16)
    q = query.to(torch.float16) if needs_cast else query
    k = key.to(torch.float16) if needs_cast else key
    v = value.to(torch.float16) if needs_cast else value
    attn_mask = kwargs.get("attn_mask", kwargs.get("attn_bias", None))
    if attn_mask is not None and attn_mask.dtype == torch.bfloat16:
        attn_mask = attn_mask.to(torch.float16)
    sdpa_kwargs = {
        "dropout_p": kwargs.get("dropout_p", 0.0),
        "is_causal": kwargs.get("is_causal", False)
    }
    if attn_mask is not None:
        sdpa_kwargs["attn_mask"] = attn_mask
    if kwargs.get("scale") is not None:
        sdpa_kwargs["scale"] = kwargs["scale"]
    q_p, k_p, v_p = q.permute(0, 2, 1, 3), k.permute(0, 2, 1, 3), v.permute(0, 2, 1, 3)
    out_p = torch.nn.functional.scaled_dot_product_attention(q_p, k_p, v_p, **sdpa_kwargs)
    out = out_p.permute(0, 2, 1, 3)
    return out.to(orig_dtype) if needs_cast else out

def safe_sliced_attention(query, key, value, *args, **kwargs):
    orig_dtype = query.dtype
    needs_cast = (query.dtype == torch.bfloat16)
    q = query.to(torch.float16) if needs_cast else query
    k = key.to(torch.float16) if needs_cast else key
    v = value.to(torch.float16) if needs_cast else value
    attn_mask = kwargs.get("attn_mask", kwargs.get("attn_bias", None))
    if attn_mask is not None and attn_mask.dtype == torch.bfloat16:
        attn_mask = attn_mask.to(torch.float16)
    sdpa_kwargs = {
        "dropout_p": kwargs.get("dropout_p", 0.0),
        "is_causal": kwargs.get("is_causal", False)
    }
    if attn_mask is not None:
        sdpa_kwargs["attn_mask"] = attn_mask
    if kwargs.get("scale") is not None:
        sdpa_kwargs["scale"] = kwargs["scale"]
    try:
        q_p, k_p, v_p = q.permute(0, 2, 1, 3), k.permute(0, 2, 1, 3), v.permute(0, 2, 1, 3)
        out_p = torch.nn.functional.scaled_dot_product_attention(q_p, k_p, v_p, **sdpa_kwargs)
        out = out_p.permute(0, 2, 1, 3)
        return out.to(orig_dtype) if needs_cast else out
    except (torch.cuda.OutOfMemoryError, RuntimeError):
        torch.cuda.empty_cache()
        H = q.shape[2]
        chunks = []
        for i in range(H):
            qi_p = q[:, :, i:i+1, :].permute(0, 2, 1, 3)
            ki_p = k[:, :, i:i+1, :].permute(0, 2, 1, 3)
            vi_p = v[:, :, i:i+1, :].permute(0, 2, 1, 3)
            chunk_p = torch.nn.functional.scaled_dot_product_attention(qi_p, ki_p, vi_p, **sdpa_kwargs)
            chunks.append(chunk_p.permute(0, 2, 1, 3))
        out = torch.cat(chunks, dim=2)
        return out.to(orig_dtype) if needs_cast else out

def setup_attention_dispatcher(model_choice: str):
    active_fn = fast_full_attention if model_choice == "flux_klein_4b" else safe_sliced_attention
    attn_dispatch._native_attention = active_fn
    attn_dispatch.dispatch_attention_fn = active_fn
    if hasattr(attn_dispatch, "_AttentionBackendRegistry"):
        for b_name in list(attn_dispatch._AttentionBackendRegistry._backends.keys()):
            attn_dispatch._AttentionBackendRegistry._backends[b_name] = active_fn
        attn_dispatch._AttentionBackendRegistry.set_active_backend(attn_dispatch.AttentionBackendName.NATIVE)

setup_attention_dispatcher(MODEL_CHOICE)

MODEL_CONFIGS = {
    "flux_klein_4b": {
        "repo_id": "Disty0/FLUX.2-klein-4B-SDNQ-4bit-dynamic",
        "embeds_file": "flux4b_manga_embeds.pt",
        "name": "FLUX.2 Klein 4B",
    },
    "flux_klein_9b": {
        "repo_id": "Disty0/FLUX.2-klein-9B-SDNQ-4bit-dynamic-svd-r32",
        "embeds_file": "flux9b_manga_embeds.pt",
        "name": "FLUX.2 Klein 9B",
    }
}

def locate_embeds_file(filename: str) -> str:
    for path in [os.path.join(DRIVE_DIR, filename), filename, os.path.join("/content", filename)]:
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        uploaded = files.upload()
        if filename in uploaded:
            dest = os.path.join(DRIVE_DIR, filename) if os.path.exists(DRIVE_DIR) else filename
            with open(dest, "wb") as f:
                f.write(uploaded[filename])
            return dest
    except Exception:
        pass
    raise FileNotFoundError(f"Embeddings '{filename}' not found. Please upload to Google Drive root.")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
EMBEDS_CACHE = {}

def get_cached_embeds(model_key: str, device: str):
    if model_key not in EMBEDS_CACHE:
        fpath = locate_embeds_file(MODEL_CONFIGS[model_key]["embeds_file"])
        raw = torch.load(fpath, map_location=device)
        if isinstance(raw, dict):
            raw = raw.get("prompt_embeds", raw)
        if isinstance(raw, torch.Tensor):
            raw = raw.to(dtype=dtype, device=device)
        EMBEDS_CACHE[model_key] = raw
    return EMBEDS_CACHE[model_key]

app = FastAPI(title="MangaTranslator Remote Inpainting Server")
ACTIVE_PIPELINE = None
ACTIVE_MODEL_NAME = None
PIPELINE_LOCK = asyncio.Lock()
INFERENCE_LOCK = asyncio.Lock()

def build_single_pipeline(target_model: str):
    cfg = MODEL_CONFIGS[target_model]
    setup_attention_dispatcher(target_model)
    pipe = Flux2KleinPipeline.from_pretrained(
        cfg["repo_id"],
        text_encoder=None,
        tokenizer=None,
        torch_dtype=dtype,
        device_map="cuda",
        token=os.environ.get("HF_TOKEN"),
    )
    if hasattr(pipe.transformer, "set_attention_backend"):
        try:
            pipe.transformer.set_attention_backend("native")
        except Exception:
            pass

    if target_model == "flux_klein_4b":
        # 4B uses direct decoding (~2.45 GB VRAM)
        for attr in ["disable_attention_slicing", "disable_tiling", "disable_slicing"]:
            for obj in [pipe, getattr(pipe, "vae", None)]:
                if obj and hasattr(obj, attr):
                    try:
                        getattr(obj, attr)()
                    except Exception:
                        pass
    else:
        # 9B enables tiling/slicing to prevent OOM
        if hasattr(pipe, "vae") and pipe.vae is not None:
            for attr in ["enable_tiling", "enable_slicing"]:
                if hasattr(pipe.vae, attr):
                    try:
                        getattr(pipe.vae, attr)()
                    except Exception:
                        pass

    pipe._model_name = target_model
    return pipe

async def ensure_pipeline(target_model: str):
    global ACTIVE_PIPELINE, ACTIVE_MODEL_NAME
    if ACTIVE_PIPELINE is not None and ACTIVE_MODEL_NAME == target_model:
        return ACTIVE_PIPELINE
    async with PIPELINE_LOCK:
        if ACTIVE_PIPELINE is not None and ACTIVE_MODEL_NAME == target_model:
            return ACTIVE_PIPELINE
        if ACTIVE_PIPELINE is not None:
            del ACTIVE_PIPELINE
            ACTIVE_PIPELINE = None
            gc.collect()
            torch.cuda.empty_cache()

        cfg = MODEL_CONFIGS[target_model]
        pipe = await asyncio.to_thread(build_single_pipeline, target_model)
        ACTIVE_PIPELINE = pipe
        ACTIVE_MODEL_NAME = target_model
        vram_used = round(torch.cuda.memory_allocated(0) / (1024**3), 2)
        print(f"✅ Loaded {cfg['name']} (VRAM: {vram_used} GB)")
        return ACTIVE_PIPELINE

print(f"⏳ Pre-loading {MODEL_CONFIGS[MODEL_CHOICE]['name']}...")
ACTIVE_PIPELINE = build_single_pipeline(MODEL_CHOICE)
ACTIVE_MODEL_NAME = MODEL_CHOICE
get_cached_embeds(MODEL_CHOICE, device)
vram_used = round(torch.cuda.memory_allocated(0) / (1024**3), 2)
print(f"✅ Ready: {MODEL_CONFIGS[MODEL_CHOICE]['name']} (VRAM: {vram_used} GB)")

@app.get("/")
@app.get("/health")
def health_check():
    vram_gb = round(torch.cuda.memory_allocated(0) / (1024**3), 2) if torch.cuda.is_available() else 0
    return {
        "status": "ok",
        "loaded_model": MODEL_CONFIGS[ACTIVE_MODEL_NAME]["name"] if ACTIVE_MODEL_NAME else MODEL_CONFIGS[MODEL_CHOICE]["name"],
        "vram_gb": vram_gb
    }

@app.post("/inpaint")
async def inpaint_endpoint(
    image: UploadFile = File(...),
    mask: UploadFile = File(None),
    prompt: str = Form(""),
    width: int = Form(1024),
    height: int = Form(1024),
    num_inference_steps: int = Form(4),
    guidance_scale: float = Form(1.0),
    seed: int = Form(-1),
    model_name: str = Form(None)
):
    try:
        active_model = model_name if (model_name and model_name in MODEL_CONFIGS) else MODEL_CHOICE
        img_bytes = await image.read()
        img_pil = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        orig_w, orig_h = img_pil.size
        target_w = max(64, min(1024, round(orig_w / 16) * 16))
        target_h = max(64, min(1024, round(orig_h / 16) * 16))
        img_in = img_pil.resize((target_w, target_h), Image.Resampling.LANCZOS) if (target_w, target_h) != (orig_w, orig_h) else img_pil

        pipe = await ensure_pipeline(active_model)

        async with INFERENCE_LOCK:
            embeds = get_cached_embeds(active_model, device)
            generator = torch.Generator(device="cpu").manual_seed(int(seed)) if (seed is not None and int(seed) >= 0) else None

            pipe_kwargs = {
                "image": img_in,
                "height": target_h,
                "width": target_w,
                "guidance_scale": float(guidance_scale),
                "num_inference_steps": max(1, min(12, int(num_inference_steps))),
                "generator": generator,
                "prompt_embeds": embeds.to(device),
            }

            def _run_gpu_inference():
                start_t = time.time()
                with torch.inference_mode():
                    out = pipe(**pipe_kwargs)
                return out.images[0], round(time.time() - start_t, 2)

            result_img, elapsed = await asyncio.to_thread(_run_gpu_inference)
            if (target_w, target_h) != (orig_w, orig_h):
                result_img = result_img.resize((orig_w, orig_h), Image.Resampling.LANCZOS)

            out_buf = io.BytesIO()
            result_img.save(out_buf, format="PNG")
            print(f"[Inpaint] {orig_w}x{orig_h} in {elapsed}s")
            return Response(content=out_buf.getvalue(), media_type="image/png")
    except Exception as e:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return JSONResponse(status_code=500, content={"error": str(e)})

@app.get("/reset")
@app.post("/reset")
def reset_endpoint():
    global ACTIVE_PIPELINE, ACTIVE_MODEL_NAME
    ACTIVE_PIPELINE = None
    ACTIVE_MODEL_NAME = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    return {"status": "ok"}

SERVER_PORT = 8000
if not os.path.exists("/usr/local/bin/cloudflared"):
    os.system("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")

async def run_server():
    server = uvicorn.Server(uvicorn.Config(app=app, host="127.0.0.1", port=SERVER_PORT, log_level="warning"))
    server_task = asyncio.create_task(server.serve())
    await asyncio.sleep(1.5)

    cf_log = "/tmp/cloudflared.log"
    os.system(f"rm -f {cf_log}")
    proc = subprocess.Popen([
        "cloudflared", "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}", "--protocol", "http2", "--logfile", cf_log
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    tunnel_url = None
    try:
        for _ in range(40):
            await asyncio.sleep(0.5)
            if os.path.exists(cf_log):
                with open(cf_log, "r") as f:
                    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
                    if match:
                        tunnel_url = match.group(0)
                        break
            if proc.poll() is not None:
                break
    except (asyncio.CancelledError, KeyboardInterrupt):
        proc.terminate()
        os.system("pkill -9 cloudflared 2>/dev/null || true")
        raise

    if not tunnel_url:
        proc.terminate()
        os.system("pkill -9 cloudflared 2>/dev/null || true")
        raise RuntimeError("Failed to obtain Cloudflare tunnel URL.")

    print(f"⏳ Checking connection ({tunnel_url}/health)...")
    health_ok = False
    def _check_health():
        try:
            req = urllib.request.Request(f"{tunnel_url}/health", headers={"User-Agent": "MangaTranslator-Check/1.0"})
            with urllib.request.urlopen(req, timeout=3) as resp:
                return resp.status == 200
        except Exception:
            return False

    for _ in range(30):
        if await asyncio.to_thread(_check_health):
            health_ok = True
            break
        await asyncio.sleep(1.0)

    print("\n" + "=" * 55)
    print(f"🎉 SERVER READY ({MODEL_CONFIGS[MODEL_CHOICE]['name']})!")
    print(f"🔗 URL: {tunnel_url}")
    if not health_ok:
        print("⚠️ Note: Cloudflare edge is still propagating.")
    print("=" * 55 + "\n")

    try:
        await server_task
    except (KeyboardInterrupt, SystemExit, asyncio.CancelledError):
        pass
    finally:
        server.should_exit = True
        proc.terminate()
        os.system("pkill -9 cloudflared 2>/dev/null || true")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

await run_server()